In [ ]:
# Section 1: Feature Extraction (English dataset)
import pandas as pd
import spacy
from empath import Empath
from collections import Counter
import os

# Load Data
df = pd.read_csv("data/final_eng.csv")

print("Rows loaded:", len(df))
print(df.head())

if len(df) == 0:
    raise ValueError("Dataset is empty")

# Initialize tools
nlp = spacy.load("en_core_web_sm")
lexicon = Empath()

# Feature Extraction
features = []

for _, row in df.iterrows():
    text = str(row['sentence'])
    doc = nlp(text)

    # Word counts
    valid_tokens = [t for t in doc if t.is_alpha]
    word_count = len(valid_tokens)
    unique_words = len(set([t.text.lower() for t in valid_tokens]))

    # POS normalized percentages
    pos_counts = Counter([t.pos_ for t in valid_tokens])
    total_tokens = len(valid_tokens)

    if total_tokens == 0:
        noun_pct = verb_pct = adj_pct = adv_pct = 0.0
    else:
        noun_pct = pos_counts.get("NOUN", 0) / total_tokens
        verb_pct = pos_counts.get("VERB", 0) / total_tokens
        adj_pct  = pos_counts.get("ADJ", 0)  / total_tokens
        adv_pct  = pos_counts.get("ADV", 0)  / total_tokens

    # Empath categories
    empath_features = lexicon.analyze(text, normalize=True)

    # Combine features
    row_features = {
        "sentence": text,
        "label": row["label"],
        "distortion": row["distortion"],
        "word_count": word_count,
        "unique_words": unique_words,
        "noun_pct": noun_pct,
        "verb_pct": verb_pct,
        "adj_pct": adj_pct,
        "adv_pct": adv_pct,
        **empath_features
    }

    features.append(row_features)

# Save
output_dir = "data/final_outputs"
os.makedirs(output_dir, exist_ok=True)

features_df = pd.DataFrame(features)
features_df.to_csv(f"{output_dir}/eng_spacy_empath_raw.csv", index=False)

print("Done. English features saved.")
print("Output rows:", len(features_df))

In [ ]:
# Section 2: Empath Feature Analysis
import pandas as pd
from scipy.stats import ttest_ind
import os

# Load dataset
df = pd.read_csv("data/final_outputs/eng_spacy_empath_raw.csv")

print("Total rows:", len(df))
if len(df) == 0:
    raise ValueError("Dataset is empty.")

# Identify Empath columns
exclude_cols = [
    'sentence', 'label', 'distortion',
    'word_count', 'unique_words',
    'noun_pct', 'verb_pct', 'adj_pct', 'adv_pct'
]

empath_cols = [c for c in df.columns if c not in exclude_cols]

# Clean distortion column
df = df.dropna(subset=["distortion"])

# Split groups
distorted_df = df[df['distortion'] != 'No Distortion']
non_distorted_df = df[df['distortion'] == 'No Distortion']

print("Distorted:", len(distorted_df))
print("Non-distorted:", len(non_distorted_df))

if len(distorted_df) == 0 or len(non_distorted_df) == 0:
    raise ValueError("One of the groups is empty")

# Compute means
mean_distorted = distorted_df[empath_cols].mean()
mean_non_distorted = non_distorted_df[empath_cols].mean()

summary = pd.DataFrame({
    "mean_distorted": mean_distorted,
    "mean_non_distorted": mean_non_distorted,
})

# T-test per category
p_values = []

for col in empath_cols:
    d1 = distorted_df[col].dropna()
    d2 = non_distorted_df[col].dropna()

    if len(d1) < 2 or len(d2) < 2:
        p_values.append(None)
        continue

    _, p = ttest_ind(d1, d2, equal_var=False)
    p_values.append(p)

summary["p_value"] = p_values
summary["significant"] = summary["p_value"] < 0.05
summary["diff"] = summary["mean_distorted"] - summary["mean_non_distorted"]

# Sort
summary_sorted = summary.sort_values(by="diff", ascending=False)

# Save
os.makedirs("data/final_outputs", exist_ok=True)
output_path = "data/final_outputs/eng_empath_distorted_vs_non.csv"
summary_sorted.to_csv(output_path)

print("Saved:", output_path)

In [ ]:
# Section 3: POS + Word Count Feature Analysis
import pandas as pd
from scipy.stats import ttest_ind
import os

# Load dataset
df = pd.read_csv("data/final_outputs/eng_spacy_empath_raw.csv")

print("Total rows:", len(df))
if len(df) == 0:
    raise ValueError("Dataset is empty.")

# POS + lexical features
analysis_cols = ['noun_pct', 'verb_pct', 'adj_pct', 'adv_pct', 'word_count']

# Clean data
df = df.dropna(subset=["distortion"])

# Split groups
distorted_df = df[df['distortion'] != 'No Distortion']
non_distorted_df = df[df['distortion'] == 'No Distortion']

print("Distorted:", len(distorted_df))
print("Non-distorted:", len(non_distorted_df))

if len(distorted_df) == 0 or len(non_distorted_df) == 0:
    raise ValueError("One of the groups is empty")

# Means
mean_distorted = distorted_df[analysis_cols].mean()
mean_non_distorted = non_distorted_df[analysis_cols].mean()

summary = pd.DataFrame({
    "mean_distorted": mean_distorted,
    "mean_non_distorted": mean_non_distorted,
})

# T-tests
p_values = []

for col in analysis_cols:
    d1 = distorted_df[col].dropna()
    d2 = non_distorted_df[col].dropna()

    if len(d1) < 2 or len(d2) < 2:
        p_values.append(None)
        continue

    _, p = ttest_ind(d1, d2, equal_var=False)
    p_values.append(p)

summary["p_value"] = p_values
summary["significant"] = summary["p_value"] < 0.05
summary["diff"] = summary["mean_distorted"] - summary["mean_non_distorted"]

# Sort
summary_sorted = summary.sort_values(by="diff", ascending=False)

# Save
os.makedirs("data/final_outputs", exist_ok=True)
output_path = "data/final_outputs/eng_pos_wordcount_distorted_vs_non.csv"
summary_sorted.to_csv(output_path)

print("Saved:", output_path)

In [ ]:
# Section 4: Top Empath Categories per Distortion Type
import pandas as pd
import os

# Load dataset
df = pd.read_csv("data/final_outputs/eng_spacy_empath_raw.csv")

print("Total rows:", len(df))
if len(df) == 0:
    raise ValueError("Dataset is empty.")

# Identify Empath columns
exclude_cols = [
    'sentence', 'label', 'distortion',
    'word_count', 'unique_words',
    'noun_pct', 'verb_pct', 'adj_pct', 'adv_pct'
]

empath_cols = [c for c in df.columns if c not in exclude_cols]

# Clean distortion column
df = df.dropna(subset=["distortion"])

# Filter distorted texts only
distorted_df = df[df['distortion'] != 'No Distortion']

print("Distorted rows:", len(distorted_df))

if len(distorted_df) == 0:
    raise ValueError("No distorted data found.")

# Average category scores per distortion type
distortion_types = distorted_df['distortion'].unique()

per_type_summary = {}

for dtype in distortion_types:
    subset = distorted_df[distorted_df['distortion'] == dtype]
    per_type_summary[dtype] = subset[empath_cols].mean()

per_type_df = pd.DataFrame(per_type_summary).T

# Top N categories per distortion type
top_n = 5
top_categories_per_type = {}

for dtype in per_type_df.index:
    sorted_categories = per_type_df.loc[dtype].sort_values(ascending=False)
    top_categories_per_type[dtype] = sorted_categories.head(top_n)


top_categories_df = pd.DataFrame(top_categories_per_type)

# Save
os.makedirs("data/final_outputs", exist_ok=True)
output_path = "data/final_outputs/eng_empath_categories_per_distortion.csv"
top_categories_df.to_csv(output_path)

print("Saved:", output_path)